<a href="https://colab.research.google.com/github/OJB-Quantum/Monte-Carlo-Sim/blob/main/Monte_Carlo_Sim_Fib_Fractal_Res_S21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qq cupy-cuda12x matplotlib scipy gdstk > /dev/null 2>&1

In [2]:
# ── Cell 1 ─────────────────────────────────────────────────────────────────
# Global design constants — single source of truth for the entire notebook
# -------------------------------------------------------------------------

import numpy as np

FREQ_MIN_GHZ, FREQ_MAX_GHZ = 5.0, 8.0          # design band (GHz)
NUM_RESONATORS = 9                             # total Fibonacci limbs

# Evenly spaced target peaks
freqs = np.linspace(FREQ_MIN_GHZ,
                    FREQ_MAX_GHZ,
                    NUM_RESONATORS)

print(f"[Cell 1] Design grid initialised → {freqs} GHz")

[Cell 1] Design grid initialised → [5.    5.375 5.75  6.125 6.5   6.875 7.25  7.625 8.   ] GHz


In [3]:
# ── Cell 2 ───────────────────────────────────────────────────────────────────
# Fibonacci-meander geometry & quarter-wave scaling
# ---------------------------------------------------------------------------

import numpy as np

# Safety guard: make sure Cell 1 ran
required = ['FREQ_MIN_GHZ', 'FREQ_MAX_GHZ', 'NUM_RESONATORS', 'freqs']
missing  = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Cell 1 must run first; missing globals: {missing}")

# ── Design parameters
# Calibrated quarter-wave length @ F_REF_GHZ (empirically 5 GHz device)
F_REF_GHZ          = FREQ_MIN_GHZ          # keep symbolic
LQ_REF_GHZ_UM = 3822.0   # ← decrease this to push every pole upward,
                         #    or increase it to pull them down.

# Line-width / gap envelopes (linear taper across the band)
W_MIN_UM, W_MAX_UM = 1.0, 3.0              # centre strip width range
G_MIN_UM, G_MAX_UM = 0.6, 2.0              # gap range

w_array = np.linspace(W_MIN_UM, W_MAX_UM, NUM_RESONATORS)
g_array = np.linspace(G_MIN_UM, G_MAX_UM, NUM_RESONATORS)

def choose_w(index: int) -> float:
    """Return CPW centre-strip width for resonator *index* (µm)."""
    return w_array[index]

def choose_g(index: int) -> float:
    """Return CPW gap for resonator *index* (µm)."""
    return g_array[index]

# ── λ/4 electrical length (guided) as a function of frequency
def lambda_q(f_GHz: float) -> float:
    """
    Guided quarter-wavelength in µm.
    Scales inversely with frequency, anchored at F_REF_GHZ.
    """
    return LQ_REF_GHZ_UM * (F_REF_GHZ / f_GHz)

# Fast sanity check
print(
    f"[Cell 2]  λ_q(5 GHz) = {lambda_q(5.0):.0f} µm | "
    f"λ_q(8 GHz) = {lambda_q(8.0):.0f} µm"
)

[Cell 2]  λ_q(5 GHz) = 3822 µm | λ_q(8 GHz) = 2389 µm


In [4]:
# ── Cell 3 ───────────────────────────────────────────────────────────────────
# Extract lumped inductance (L) and capacitance (C) for every λ/4 resonator
# ----------------------------------------------------------------------------

import numpy as np
import pandas as pd
from math import pi, sqrt
from scipy.special import ellipk

# ── Guards
REQUIRED = ['freqs', 'choose_w', 'choose_g', 'lambda_q', 'NUM_RESONATORS']
missing  = [sym for sym in REQUIRED if sym not in globals()]
if missing:
    raise RuntimeError(f"Run Cells 1 & 2 first; missing -> {missing}")

# ── Physical constants (ASCII names)
EPS0 = 8.854_187_817e-12        # F m⁻¹
MU0  = 4 * pi * 1e-7            # H m⁻¹

# ── Process parameters
EPS_R_SUBSTRATE = 11.45
EPS_EFF = (EPS_R_SUBSTRATE + 1) / 2
FILM_T_NM      = 150.0          # film thickness
LAMBDA_L0_NM   = 95.0           # London depth

# ── Helper: kinetic-inductance fraction L_k/L_g
def lk_fraction(w_um: float, g_um: float) -> float:
    # --- new guard
    if g_um <= 0.0:
        raise ValueError("Gap must be > 0 µm to avoid k = 1 singularity.")
    k_raw = w_um / (w_um + 2.0 * g_um)
    k = min(k_raw, 0.999)          # cap k so ellipk() stays finite
    # ----------------------

    t_m   = FILM_T_NM * 1e-9
    lam_m = LAMBDA_L0_NM * 1e-9
    w_m   = w_um * 1e-6
    g_m   = g_um * 1e-6
    k_p   = sqrt(1.0 - k*k)
    L_geo = (MU0/4.0) * ellipk(k_p**2) / ellipk(k**2)
    L_kin = MU0 * lam_m**2 / (w_m * t_m)
    return L_kin / L_geo

# ── Allocate result arrays (SI units)
L_total_H = np.empty(NUM_RESONATORS)
C_total_F = np.empty(NUM_RESONATORS)

# ── Main loop
for idx, f_GHz in enumerate(freqs):
    w_um, g_um = choose_w(idx), choose_g(idx)
    w_m, g_m   = w_um*1e-6, g_um*1e-6
    k = w_m / (w_m + 2*g_m)
    k_p = sqrt(1 - k*k)

    L_geo = (MU0/4) * ellipk(k_p**2) / ellipk(k**2)
    C_geo = 4*EPS0*EPS_EFF * ellipk(k**2) / ellipk(k_p**2)

    frac = lk_fraction(w_um, g_um)
    L_prime = L_geo * (1 + frac)

    lam_q_um   = lambda_q(f_GHz) / sqrt(1 + frac)
    res_len_m  = lam_q_um * 1e-6

    L_total_H[idx] = L_prime * res_len_m
    C_total_F[idx] = C_geo   * res_len_m

# ── Build result table
res_df = pd.DataFrame({
    "f_target_GHz": freqs,
    "w_um": [choose_w(i) for i in range(NUM_RESONATORS)],
    "g_um": [choose_g(i) for i in range(NUM_RESONATORS)],
    "L_H":  L_total_H,
    "C_F":  C_total_F,
})
res_df["L_total_pH"] = res_df["L_H"] * 1e12
res_df["C_total_fF"] = res_df["C_F"] * 1e15

display(res_df.style.format({
    "L_total_pH": "{:.2f}",
    "C_total_fF": "{:.2f}",
    "w_um": "{:.2f}",
    "g_um": "{:.2f}",
}))

print("[Cell 3] Lumped-element table generated.")

,f_target_GHz,w_um,g_um,L_H,C_F,L_total_pH,C_total_fF
0,5.000000,1.00,0.60,0.000000,0.000000,1757.67,575.63
1,5.375000,1.25,0.77,0.000000,0.000000,1624.49,538.94
2,5.750000,1.50,0.95,0.000000,0.000000,1511.59,506.11
3,6.125000,1.75,1.12,0.000000,0.000000,1414.21,476.75
4,6.500000,2.00,1.30,0.000000,0.000000,1329.11,450.44
5,6.875000,2.25,1.48,0.000000,0.000000,1253.98,426.76
6,7.250000,2.50,1.65,0.000000,0.000000,1187.09,405.38
7,7.625000,2.75,1.82,0.000000,0.000000,1127.11,385.99
8,8.000000,3.00,2.00,0.000000,0.000000,1073.00,368.33


[Cell 3] Lumped-element table generated.


In [5]:
# ── Sanity probe: are any f0 values NaN/Inf?
bad = res_df[res_df["L_H"].isna() | res_df["C_F"].isna() |
             ~np.isfinite(res_df["L_H"]) | ~np.isfinite(res_df["C_F"])]
print("Rows with non-finite L or C\n", bad if not bad.empty else "— none —")

f0_check = 1/(2*np.pi*np.sqrt(res_df["L_H"]*res_df["C_F"])) / 1e9
print("\nCalculated f0 (GHz):\n", f0_check.values)

Rows with non-finite L or C
 — none —

Calculated f0 (GHz):
 [5.00357569 5.37884387 5.75411205 6.12938022 6.5046484  6.87991658
 7.25518475 7.63045293 8.00572111]


In [6]:
# ── Cell 4 ────────────────────────────────────────────────────────────────────
# Deterministic |S21| prediction — aligned with Cells 1-3
# ---------------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt; import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 300

# ── Guards
NEEDED = ['freqs', 'L_total_H', 'C_total_F', 'FREQ_MIN_GHZ', 'FREQ_MAX_GHZ']
missing = [x for x in NEEDED if x not in globals() and
           not (x == 'L_total_H' and 'L_total_pH' in globals())]
if missing:
    raise RuntimeError(f"Run Cells 1–3 first; missing → {missing}")

# ── CuPy / NumPy shim
try:
    import cupy as cp                           # GPU backend
    xp = cp
    _gpu = True
except ModuleNotFoundError:                     # CPU fallback
    xp = np                                     # type: ignore
    _gpu = False

# ── Import lumped elements (SI units)
if 'L_total_pH' in globals():                   # pH / fF → H / F
    L0_H = np.asarray(L_total_pH) * 1e-12
    C0_F = np.asarray(C_total_fF) * 1e-15
else:
    L0_H = np.asarray(L_total_H)
    C0_F = np.asarray(C_total_F)

# ── Loaded-Q mapping
if 'QL_freq_lookup' in globals():
    QL_arr = np.array([QL_freq_lookup.get(float(f), 1.0e4) for f in freqs])
else:                                           # default fallback
    QL_arr = np.full(len(freqs), 1.0e4)

# ── Resonant frequencies (GHz) on selected backend
L0_cp, C0_cp = xp.asarray(L0_H), xp.asarray(C0_F)
f0_cp_GHz = 1.0 / (2.0 * xp.pi * xp.sqrt(L0_cp * C0_cp)) / 1e9
f0_calc_GHz = xp.asnumpy(f0_cp_GHz) if _gpu else f0_cp_GHz

# ── Frequency sweep grid
MARGIN_GHZ = 0.5
N_POINTS   = 250_000
f_grid_GHz = xp.linspace(FREQ_MIN_GHZ - MARGIN_GHZ,
                         FREQ_MAX_GHZ + MARGIN_GHZ,
                         N_POINTS, dtype=xp.float64)
f_grid_Hz  = f_grid_GHz * 1e9

# ── Cascade shunt-notch poles
S21_cp = xp.ones_like(f_grid_Hz, dtype=xp.complex128)

for f0_GHz, QL in zip(f0_cp_GHz, QL_arr):
    x = f_grid_Hz / (f0_GHz * 1e9) - 1.0
    H = 1.0 - 1.0 / (1.0 + 2.0j * QL * x)
    S21_cp *= H

S21_dB = 20.0 * xp.log10(xp.abs(S21_cp))
S21_dB_cpu = xp.asnumpy(S21_dB) if _gpu else S21_dB
f_plot = xp.asnumpy(f_grid_GHz) if _gpu else f_grid_GHz

# ── Quick peak-alignment report
print("[Cell 4] Δ_peak (MHz):", (f0_calc_GHz - freqs) * 1e3)

# ── Plot
plt.figure(figsize=(10, 6))
plt.plot(f_plot, S21_dB_cpu, color="blue", lw=1.5)
plt.xlabel("Frequency (GHz)")
plt.ylabel("|S\u2082\u2081| (dB)")          # subscript unicode
plt.title("Predicted Transmission — Fibonacci λ/4 CPW Resonators")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.ylim(-50, 2)
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [ ]:
# Cell 4-MC — True Monte-Carlo |S21|  (auto-calibrated to design freqs)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib as mpl
mpl.rcParams["figure.dpi"] = 250

# ── GPU backend (CuPy) with CPU fallback
try:
    import cupy as cp
    xp, _gpu = cp, True
except ModuleNotFoundError:
    xp, _gpu = np, False

# ── Global checks
NEEDED = ['freqs', 'FREQ_MIN_GHZ', 'FREQ_MAX_GHZ']
_missing = [k for k in NEEDED if k not in globals()]
if _missing:
    raise RuntimeError(f"Run Cells 1-3 first; missing → {_missing}")

# ── Lumped elements with unit-robust import
def pick_array(prefix_si: str, prefix_pico: str, scale: float):
    if prefix_si in globals():
        return np.asarray(globals()[prefix_si], float)
    if prefix_pico in globals():
        return np.asarray(globals()[prefix_pico], float) * scale
    raise NameError(f"Need '{prefix_si}' or '{prefix_pico}'.")

L0_H = pick_array('L_total_H', 'L_total_pH', 1e-12)     # pH → H
C0_F = pick_array('C_total_F', 'C_total_fF', 1e-15)     # fF → F
N_RES = len(L0_H)

# ── **Auto-calibration**  (aligns f0_calc to design freqs)
f0_calc_GHz = 1.0 / (2*np.pi*np.sqrt(L0_H * C0_F)) / 1e9
ratio = f0_calc_GHz / np.asarray(freqs, float)          # >1 → sim too high

# apply per-resonator correction to L (you could equivalently scale C)
L0_H = L0_H * ratio**2          # increase L when sim freq > design freq
# optional: print residual error
resid = (1.0 / (2*np.pi*np.sqrt(L0_H * C0_F)) / 1e9) - np.asarray(freqs)
print("[cal]  max |f_calc-f_design| after calib:",
      f"{np.max(np.abs(resid))*1e3:.2f} MHz")

# ── Loaded-Q array
if 'QL_arr' in globals():
    QL_arr = np.asarray(QL_arr, float)
    if QL_arr.size == 1:
        QL_arr = np.full(N_RES, QL_arr.item())
else:
    QL_arr = np.full(N_RES, 10_000.0)

# ── Monte-Carlo knobs
NUM_PARTICLES    = 50_000
CHUNK_PARTICLES  = 1_024
SIGMA_L, SIGMA_C = 0.02, 0.02
COUPLING_DEPTH   = 0.85
INSERTION_LOSS   = 0.04
SEED             = 123
(xp.random if _gpu else np.random).seed(SEED)

# ── Frequency sweep grid
SHIFT_GHZ        = 0.20                     # ← patch
SWEEP_MIN_GHZ    = 4.5
SWEEP_MAX_GHZ    = 8.5
N_POINTS         = 250_000

#  We generate the raw grid before the shift, so that after adding
#  +SHIFT_GHZ during plotting the visible axis is 4.5–8.5 GHz.
f_grid_GHz = xp.linspace(SWEEP_MIN_GHZ - SHIFT_GHZ,
                         SWEEP_MAX_GHZ - SHIFT_GHZ,
                         N_POINTS, dtype=xp.float64)
f_grid_Hz  = f_grid_GHz * 1e9

# ── Notch transfer helper
def notch_mag(f_Hz, f0_Hz, Q_L):
    x    = f_Hz / f0_Hz - 1.0
    peak = 1.0 / xp.sqrt(1.0 + (2.0 * Q_L * x)**2)
    return (1.0 - COUPLING_DEPTH * peak) * (1.0 - INSERTION_LOSS)

# ── Monte-Carlo streaming
S21_min = xp.full(N_POINTS, xp.inf, dtype=xp.float64)

for start in range(0, NUM_PARTICLES, CHUNK_PARTICLES):
    end  = min(start + CHUNK_PARTICLES, NUM_PARTICLES)
    blk  = end - start

    dL = xp.random.standard_normal((blk, N_RES)) * SIGMA_L
    dC = xp.random.standard_normal((blk, N_RES)) * SIGMA_C
    Lb = xp.asarray(L0_H) * (1 + dL)
    Cb = xp.asarray(C0_F) * (1 + dC)
    f0 = 1.0 / (2.0 * xp.pi * xp.sqrt(Lb * Cb))  # (blk,res) [Hz]

    S_blk = xp.ones((blk, N_POINTS), dtype=xp.float64)
    for r in range(N_RES):
        S_blk *= notch_mag(f_grid_Hz[None, :], f0[:, r, None], QL_arr[r])

    S21_min = xp.minimum(S21_min, xp.min(S_blk, axis=0))

# ── Ship to CPU + dB plot
f_cpu = xp.asnumpy(f_grid_GHz) if _gpu else f_grid_GHz
S_dB  = 20*np.log10(xp.asnumpy(S21_min) if _gpu else S21_min)

plt.figure(figsize=(10, 6))
plt.plot(f_cpu + SHIFT_GHZ, S_dB, 'k', lw=2)          # apply +0.20 GHz offset
plt.ylim(-25, 0)
plt.xlim(SWEEP_MIN_GHZ, SWEEP_MAX_GHZ)                 # hard limits 4.5–8.5
plt.xlabel("Frequency (GHz)")
plt.ylabel("|S\u2082\u2081| (dB)")
plt.title("Monte-Carlo Transmission — Fibonacci λ/4 CPW Resonators\n"
          f"(x-axis shifted +{SHIFT_GHZ} GHz)")
plt.grid(ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4-MC-Bezier — overlay a composite-Bezier fit on the Monte-Carlo envelope
# ─────────────────────────────────────────────────────────────────────────────
"""
Prerequisite: run *Cell 4-MC* first so the following arrays exist in memory

    f_cpu        – sweep frequencies *before* SHIFT_GHZ offset   (NumPy 1-D)
    S_dB         – Monte-Carlo min-envelope in dB               (NumPy 1-D)
    SHIFT_GHZ    – x-axis offset applied during plotting
    SWEEP_MIN_GHZ, SWEEP_MAX_GHZ – display limits (4.5 → 8.5 GHz)

This cell:

1.  Shifts x-data by `SHIFT_GHZ`
2.  Selects clustered anchor points where the curve bends the most
3.  Builds a **composite cubic Bézier** through those anchors
4.  Plots the original envelope (grey) and the Bézier fit (red)

No external libraries beyond NumPy/Matplotlib are required.
"""
import numpy as np
import matplotlib.pyplot as plt

# ── 1) Prepare data (apply +SHIFT_GHZ) -------------------------------------
x = f_cpu + SHIFT_GHZ            # GHz, length = N_POINTS
y = S_dB                         # dB, same length

# Normalise x to [0,1] for scale-independent derivative estimates
x_norm = (x - x.min()) / (x.max() - x.min())

# ── 2) Anchor selection with curvature clustering --------------------------
def get_clustered_anchor_indices(xn, yn, num_uniform=20, thresh_pct=65):
    """
    Uniformly sample `num_uniform` points + add extra anchors where
    |second-derivative| lies above the `thresh_pct` percentile.
    """
    uniform_idx = np.linspace(0, len(xn)-1, num=num_uniform, dtype=int)
    d1 = np.gradient(yn, xn)
    d2 = np.gradient(d1,  xn)
    curv = np.abs(d2)
    extra_idx = np.where(curv > np.percentile(curv, thresh_pct))[0]
    return np.sort(np.unique(np.concatenate((uniform_idx, extra_idx))))

anchor_idx = get_clustered_anchor_indices(x_norm, y,
                                          num_uniform=20, thresh_pct=65)
ax_norm = x_norm[anchor_idx]
ay      = y[anchor_idx]

# ── 3) Estimate slopes (finite-difference) ---------------------------------
def estimate_slopes(xa, ya):
    m = np.zeros_like(ya)
    for i in range(len(ya)):
        if i == 0:
            m[i] = (ya[i+1] - ya[i]) / (xa[i+1] - xa[i])
        elif i == len(ya)-1:
            m[i] = (ya[i] - ya[i-1]) / (xa[i] - xa[i-1])
        else:
            m[i] = (ya[i+1] - ya[i-1]) / (xa[i+1] - xa[i-1])
    return m

aslope = estimate_slopes(ax_norm, ay)

# ── 4) Build composite cubic Bézier curve ----------------------------------
def bezier_composite(xa, ya, ma, samples_per_seg=50):
    cx, cy = [], []
    for i in range(len(xa)-1):
        x0, y0, m0 = xa[i],   ya[i],   ma[i]
        x1, y1, m1 = xa[i+1], ya[i+1], ma[i+1]
        dx  = x1 - x0
        P0  = np.array([x0,           y0])
        P1  = np.array([x0 + dx/3.0,  y0 + (dx/3.0)*m0])
        P2  = np.array([x1 - dx/3.0,  y1 - (dx/3.0)*m1])
        P3  = np.array([x1,           y1])

        t   = np.linspace(0, 1, samples_per_seg, endpoint=False if i < len(xa)-2 else True)
        B   = (np.outer((1-t)**3, P0) +
               np.outer(3*(1-t)**2*t, P1) +
               np.outer(3*(1-t)*t**2, P2) +
               np.outer(t**3,        P3))
        cx.extend(B[:,0]); cy.extend(B[:,1])
    return np.array(cx), np.array(cy)

bz_x_norm, bz_y = bezier_composite(ax_norm, ay, aslope, samples_per_seg=60)

# Map normalised x back to GHz
bz_x = bz_x_norm * (x.max() - x.min()) + x.min()

# ── 5) Plot ----------------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(x, y,  color='0.55', lw=1.0, label="Monte-Carlo min-envelope")
plt.plot(bz_x, bz_y, color='red', lw=2.0, label="Composite Bézier fit")
plt.scatter(x[anchor_idx], y[anchor_idx], color='blue', s=15,
            label=f"{len(anchor_idx)} anchors")

plt.xlim(SWEEP_MIN_GHZ, SWEEP_MAX_GHZ)
plt.ylim(-25, 0)
plt.xlabel("Frequency (GHz)")
plt.ylabel("|S\u2082\u2081|  (dB)")
plt.title("MC Envelope with Composite Bézier Approximation")
plt.grid(ls="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4-MC-Bezier (v3)  —  envelope-hugging composite Bézier
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema

# 1) helpers (unchanged Bézier building blocks) -----------------------------
def cubic_bezier(P0, P1, P2, P3, num=200):
    t = np.linspace(0, 1, num)[:, None]
    return ((1-t)**3)*P0 + 3*((1-t)**2)*t*P1 + 3*(1-t)*(t**2)*P2 + (t**3)*P3

def compute_derivatives(x, y):
    m = np.zeros_like(y)
    for i in range(len(y)):
        if i == 0:
            m[i] = (y[i+1]-y[i]) / (x[i+1]-x[i])
        elif i == len(y)-1:
            m[i] = (y[i]-y[i-1]) / (x[i]-x[i-1])
        else:
            m[i] = (y[i+1]-y[i-1]) / (x[i+1]-x[i-1])
    return m

def bezier_from_anchors(x, y, m, pts_per_seg=200):
    xb, yb = [], []
    for i in range(len(x)-1):
        P0 = np.array([x[i],   y[i]])
        P3 = np.array([x[i+1], y[i+1]])
        dx = P3[0]-P0[0]
        P1 = np.array([P0[0] + dx/3, P0[1] + (dx/3)*m[i]])
        P2 = np.array([P3[0] - dx/3, P3[1] - (dx/3)*m[i+1]])
        seg = cubic_bezier(P0, P1, P2, P3, pts_per_seg)
        if i: seg = seg[1:]
        xb.append(seg[:,0]); yb.append(seg[:,1])
    return np.concatenate(xb), np.concatenate(yb)

# 2) prepare data  ----------------------------------------------------------
x_data = f_cpu + SHIFT_GHZ          # GHz
y_data = S_dB                       # dB (≤0)

# 3) choose anchor indices  -------------------------------------------------
#    • local minima (primary resonant dips)
min_idx = argrelextrema(y_data, np.less)[0]

#    • a few uniform anchors to guide the curve outside the dip regions
NUM_UNIFORM = 6
uni_idx = np.linspace(0, len(x_data)-1, NUM_UNIFORM, dtype=int)

#    • extra anchors only at VERY high curvature points (90 th percentile)
d1 = np.gradient(y_data, x_data)
d2 = np.gradient(d1, x_data)
curv = np.abs(d2)
high_curv_idx = np.where(curv > np.percentile(curv, 90))[0]

# combine and sort
idx = np.sort(np.unique(np.concatenate((min_idx, uni_idx, high_curv_idx))))
x_anchor, y_anchor = x_data[idx], y_data[idx]

# 4) slopes + Bézier curve --------------------------------------------------
m_anchor = compute_derivatives(x_anchor, y_anchor)
xb, yb   = bezier_from_anchors(x_anchor, y_anchor, m_anchor, pts_per_seg=200)

# 5) plot -------------------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(x_data, y_data, color="0.85", lw=1, label="MC min-envelope")
plt.plot(xb, yb, 'r', lw=2, label="Composite Bézier fit")
plt.scatter(x_anchor, y_anchor, c='k', s=18, zorder=5,
            label=f"{len(x_anchor)} anchor pts")

plt.xlim(SWEEP_MIN_GHZ, SWEEP_MAX_GHZ)
plt.ylim(-25, 0)
plt.xlabel("Frequency (GHz)")
plt.ylabel("|S\u2082\u2081|  (dB)")
plt.title("Envelope-hugging Bézier Approx. of Monte-Carlo |S21|")
plt.grid(ls="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — v18 •  Impedance-aware gap & tapered-width Fibonacci resonators
# ───────────────────────────────────────────────────────────────────────────────
# CPW feed + rotated Fibonacci-fractal λ/4 metamaterial resonators (no IDCs)
# Author : Onri Jay Benally          Last edit : 2025-07-13
#
#  ▸ Geometry-only knobs live up front.
#  ▸ Width tapers down (1.6 → 0.4 µm) as resonance ↑ rightward.
#  ▸ Gap scales with width (g ≈ 3 w) but is clamped between 2 µm … 8 µm.
#  ▸ Fillet fraction 2 % … 18 % of local step keeps every right-angle smooth.
#
#  Run Cells 1-3 first so globals (freqs, λq-helper, etc.) exist.
# ───────────────────────────────────────────────────────────────────────────────

import math, warnings
from typing import List, Tuple

import gdstk, numpy as np
from scipy.constants import mu_0
import matplotlib.pyplot as plt; import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 1600

# ╭─ sanity guard
REQ = ["FREQ_MIN_GHZ", "FREQ_MAX_GHZ", "NUM_RESONATORS", "freqs", "lambda_q"]
_missing = [k for k in REQ if k not in globals()]
if _missing:
    raise RuntimeError(f"Run the parameter cells first; missing globals → {_missing}")

# ╭─ geometry knobs
FILLET_MIN, FILLET_MAX = 0.02, 0.18         # 2 % … 18 % of local step length
FRACTAL_SPACING_UM     = 6.0                # vert. offset from CPW feed
HORZ_PITCH_UM          = 500.0              # limb-to-limb pitch

W_RES_MAX, W_RES_MIN   = 1.6, 0.4           # left-most → right-most trace width
TARGET_G_OVER_W        = 3.0                # ~50 Ω CPW heuristic (g ≈ 3 w)
G_MIN_DRC, G_MAX_DRC   = 2.0, 8.0           # fab spacing limits (µm)

tech = dict(                   # µm  (matches process sheet)
    w_feed=10.0, g_feed=6.0, feed_len=None, w_res=W_RES_MIN,
    fractal_spacing=FRACTAL_SPACING_UM, fractal_horz_spacing=HORZ_PITCH_UM,
    pad_len=300.0, pad_wid=150.0, pad_taper_len=250.0,
    sq_margin=50.0, fillet_radius=8.0,
    g_res=G_MIN_DRC, margin=600.0
)
tech["g_max"] = max(tech["g_feed"], tech["g_res"])

# ╭─ Fibonacci word skeleton (unchanged)
_DIR = np.array([[1,0],[0,1],[-1,0],[0,-1]])
def _fib_word(n:int)->str:
    a,b="0","01"
    for _ in range(n-1): a,b=b,b+a
    return b

FIB_ITER=15
_WORD=_fib_word(FIB_ITER)
_NUM_MOVES=len(_WORD); _NUM_TURNS=_WORD.count("0")

_pts=[np.array([0.,0.])]; cur=0
for i,ch in enumerate(_WORD):
    if ch=="1": _pts.append(_pts[-1]+_DIR[cur])
    else:
        cur=(cur+(1 if i%2==0 else -1))%4
        _pts.append(_pts[-1]+_DIR[cur])
_unit=np.asarray(_pts)

rotate90 = lambda p: np.column_stack((-p[:,1], p[:,0]))
rect      = lambda x0,y0,x1,y1: gdstk.rectangle((x0,y0),(x1,y1))

# ╭─ preview helper WITHOUT polygon borders
def _preview(cell, figsize=(11,4)):
    """
    Largest polygon → ground (green). All others → metal (blue).
    Drawn with no outline (edgecolor='none').
    """
    polys = cell.polygons
    if not polys:
        print("No polygons to preview"); return

    areas = [p.area() for p in polys]
    idx_ground = int(np.argmax(areas))

    fig, ax = plt.subplots(figsize=figsize, facecolor='white')
    for i, poly in enumerate(polys):
        pts = poly.points
        if i == idx_ground:  # ground
            ax.fill(pts[:,0], pts[:,1],
                    facecolor='#a5d3a5', edgecolor='none', linewidth=0, zorder=1)
        else:                 # metal
            ax.fill(pts[:,0], pts[:,1],
                    facecolor='#6fa8dc', edgecolor='none', linewidth=0, zorder=10)

    ax.set_aspect('equal')
    ax.set_xlabel('µm'); ax.set_ylabel('µm')
    plt.tight_layout(); plt.show()

# ╭─ electrical helpers
USE_LK, MATERIAL = True, "Nb"
FILM_T_NM, LAMBDA_L0_NM = 80.0, 90.0

def _lk_frac(w_um:float, g_um:float)->float:
    if not USE_LK: return 0.0
    t = 80e-9 if MATERIAL.lower()=="nb" else FILM_T_NM*1e-9
    lam= 90e-9 if MATERIAL.lower()=="nb" else LAMBDA_L0_NM*1e-9
    w, g = w_um*1e-6, g_um*1e-6
    Lk = mu_0*lam**2/(w*t)
    import mpmath as mp
    k  = w/(w+2*g)
    Lg = (mu_0/4)*mp.ellipk(1-k**2)/mp.ellipk(k**2)
    return float(Lk/Lg)

def _step_len(lq_um, fil, lk):
    return lq_um/math.sqrt(1+lk)/(_NUM_MOVES - _NUM_TURNS*fil*(2-math.pi/2))

_choose_fil = lambda f: np.interp(f,
                                  (FREQ_MIN_GHZ, FREQ_MAX_GHZ),
                                  (FILLET_MIN,  FILLET_MAX))

# ╭─ width & gap choosers
def _choose_w(idx:int)->float:
    """Left-most (idx ≈ 0) wide; right-most narrow."""
    return W_RES_MAX - (W_RES_MAX-W_RES_MIN)*(
        freqs[idx]-FREQ_MIN_GHZ)/(FREQ_MAX_GHZ-FREQ_MIN_GHZ)

def _choose_g(idx:int)->float:
    """Impedance-aware gap with DRC clamp."""
    g = TARGET_G_OVER_W * _choose_w(idx)
    return max(min(g, G_MAX_DRC), G_MIN_DRC)

# ╭─ resonator builder
def _build_res(origin, idx, up=True):
    f   = freqs[idx]
    w   = _choose_w(idx)
    g   = _choose_g(idx)
    fil = _choose_fil(f)
    lk  = _lk_frac(w, g)
    step= _step_len(lambda_q(f), fil, lk)

    pts = rotate90(_unit*step)
    if not up: pts[:,1]*=-1
    pts += np.asarray(origin) - pts[0]
    fp  = gdstk.FlexPath(pts.tolist(), w,
                         bend_radius=fil*step, simple_path=True)
    return fp.to_polygons()

# ╭─ chip assembly
def build_chip(p:dict)->gdstk.Library:
    lib  = gdstk.Library(unit=1e-6)
    cell = lib.new_cell("MAIN")
    cond = []

    pitch     = p["fractal_horz_spacing"]
    p["feed_len"] = (NUM_RESONATORS-1)*pitch + 2*pitch
    feed_x0   = p["pad_len"] + p["pad_taper_len"]

    def _launch(x, sgn):
        w1,w2 = p["w_feed"], p["pad_wid"]
        xt, xp = x + sgn*p["pad_taper_len"], x + sgn*(p["pad_taper_len"]+p["pad_len"])
        return gdstk.Polygon([(x, w1/2), (xt, w2/2), (xp, w2/2),
                              (xp,-w2/2), (xt,-w2/2), (x,-w1/2)])

    # feed pads + line
    cell.add(_launch(feed_x0,-1), _launch(feed_x0+p["feed_len"],1))
    feed = rect(feed_x0,-p["w_feed"]/2, feed_x0+p["feed_len"], p["w_feed"]/2)
    cell.add(feed); cond.extend(cell.polygons)

    # resonators
    start_x = feed_x0 + pitch
    for i in range(NUM_RESONATORS):
        x = start_x + i*pitch
        yoff = p["w_feed"]/2 + p["g_feed"] + p["fractal_spacing"]
        y  =  yoff if i%2==0 else -yoff
        polys = _build_res((x,y), i, up=(i%2==0))
        cell.add(*polys); cond.extend(polys)

    # ground frame
    cu   = gdstk.boolean(cond, [], "or")
    xs   = np.concatenate([pl.points[:,0] for pl in cu])
    ys   = np.concatenate([pl.points[:,1] for pl in cu])
    frame= rect(xs.min()-p["margin"], ys.min()-p["margin"],
                xs.max()+p["margin"], ys.max()+p["margin"])
    keep = gdstk.offset(cu, p["g_max"]+0.1, join="round") or []
    ground = gdstk.boolean(frame, keep, "not") if keep else [frame]
    cell.add(*ground)

    return lib

# ╭─ demo run
if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    gds_lib = build_chip(tech)
    _preview(gds_lib.top_level()[0])
    gds_lib.write_oas("cpw_fibonacci_resonators_v18.oas")
    print("\nOASIS written → cpw_fibonacci_resonators_v18.oas")

In [ ]:
# Cell 6 — Monte-Carlo |S21| Prediction  ➜  dB Y-axis
# ─────────────────────────────────────────────────────────────────────────────
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

DTYPE            = cp.float32
NUM_PARTICLES    = 50_000
CHUNK_PARTICLES  = 1_024
N_SWEEP          = 200_000
MARGIN           = 0.20
KIN_LK_SIGMA     = 3e-3
G_LW_SIGMA       = 1e-3
TEMP_SIGMA       = 15e-3
SEED             = 42

# ── 1) RNG helper
cp.random.seed(SEED)
_normal = lambda mu, sig, shape: mu + sig * cp.random.standard_normal(shape, dtype=DTYPE)

# ── 2) Inputs
try:
    f_nominal_GHz = cp.asarray(freqs, dtype=DTYPE)
except NameError:
    print("[Cell 6]  Warning: 'freqs' not found – using 5–8 GHz demo array.")
    f_nominal_GHz = cp.asarray([5.0, 6.0, 7.0, 8.0], dtype=DTYPE)

N_RES = int(f_nominal_GHz.size)

if "Q_L" in globals():
    Q_L_gpu = cp.asarray(Q_L, dtype=DTYPE)
    if Q_L_gpu.ndim == 0:
        Q_L_gpu = cp.repeat(Q_L_gpu, N_RES)
else:
    Q_L_gpu = cp.full(N_RES, 5_000, dtype=DTYPE)

# ── 3) Monte-Carlo draws
_delta_Lk = _normal(0.0, KIN_LK_SIGMA, (NUM_PARTICLES, N_RES))
_delta_w  = _normal(0.0, G_LW_SIGMA,   (NUM_PARTICLES, N_RES))
_delta_T  = _normal(0.0, TEMP_SIGMA,   (NUM_PARTICLES, 1))

alpha = cp.asarray(-0.25, dtype=DTYPE)
beta  = cp.asarray(-1.8e-3, dtype=DTYPE)

f_shifted = f_nominal_GHz[None, :] * (
    1.0 - 0.5 * _delta_Lk + alpha * _delta_w + beta * _delta_T
)

# ── 4) Sweep grid
f_min   = cp.min(f_shifted) * (1 - MARGIN)
f_max   = cp.max(f_shifted) * (1 + MARGIN)
f_sweep = cp.linspace(f_min, f_max, N_SWEEP, dtype=DTYPE)

# ── 5) Lorentzian helper
def lorentz(f, f0, Q):
    r = f / f0 - 1.0
    return 1.0 / cp.sqrt(1.0 + (2.0 * Q * r) ** 2)

# ── 6) Allocate outputs
per_res_curves = cp.empty((N_RES, N_SWEEP), dtype=DTYPE)
global_env     = cp.full(N_SWEEP, cp.inf, dtype=DTYPE)

# ── 7) Streaming Monte-Carlo loop
for res_idx in range(N_RES):
    Qi      = Q_L_gpu[res_idx]
    fi_all  = f_shifted[:, res_idx]
    res_min = cp.full(N_SWEEP, cp.inf, dtype=DTYPE)

    for start in range(0, NUM_PARTICLES, CHUNK_PARTICLES):
        end      = min(start + CHUNK_PARTICLES, NUM_PARTICLES)
        fi_block = fi_all[start:end]
        S_block  = lorentz(f_sweep[:, None], fi_block[None, :], Qi)
        res_min  = cp.minimum(res_min, cp.min(S_block, axis=1))

    per_res_curves[res_idx] = res_min
    global_env = cp.minimum(global_env, res_min)

# ── 8) Plot in dB
to_dB = lambda x: 20 * np.log10(x)

fs_cpu = cp.asnumpy(f_sweep)
plt.figure(figsize=(10, 4))

for idx in range(N_RES):
    plt.plot(fs_cpu, to_dB(cp.asnumpy(per_res_curves[idx])),
             lw=0.8, label=f"Res {idx}")

plt.plot(fs_cpu, to_dB(cp.asnumpy(global_env)),
         lw=1.8, c="k", label="Global envelope")

plt.xlabel("Frequency (GHz)")
plt.ylabel("|S21|  (dB)")
plt.title("Predicted Per-Resonator |S21| Curves – Monte-Carlo (dB)")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, ncol=2, loc="upper right")
plt.tight_layout()
plt.show()

# ── 9) Optional CSV export (commented)
# import pandas as pd, os
# df = pd.DataFrame(to_dB(cp.asnumpy(per_res_curves.T)),
#                   columns=[f"Res_{i}" for i in range(N_RES)])
# df.insert(0, "GHz", fs_cpu)
# out_path = "per_resonator_S21_curves_dB.csv"
# df.to_csv(out_path, index=False)
# print(f"Saved curves → {os.path.abspath(out_path)}")

In [ ]:
# Cell 6 — Monte-Carlo Notch Simulation (realistic depth + dB plotting)
# ─────────────────────────────────────────────────────────────────────────────
"""
GPU-resident Monte-Carlo model that predicts **notch-type |S21|** for each
Fibonacci λ/4 resonator and plots the curves in dB.

Prerequisites from Cells 1–5
────────────────────────────
    freqs            – Nominal resonance frequencies [GHz]  (NumPy 1-D)
    NUM_RES0NATORS   – len(freqs)
    Q_L   (optional) – scalar or per-resonator array (defaults → 5 000)

Adjustable realism knobs
────────────────────────
Feel free to tighten or loosen COUPLING_DEPTH or INSERTION_LOSS until the simulated curve
overlays match your measured data.
    COUPLING_DEPTH   – 0 – 1   (0.85 = 15 % residual transmission at the dip)
    INSERTION_LOSS   – 0 – 1   (0.04 = 0.36 dB baseline loss)

Outputs
───────
    per_res_curves   – CuPy [N_RES, N_SWEEP] minimum envelope per resonator
    global_env       – CuPy [N_SWEEP] overall minimum envelope

"""
# ── 0) Imports & dtype
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

DTYPE            = cp.float32
NUM_PARTICLES    = 50_000
CHUNK_PARTICLES  = 1_024
N_SWEEP          = 200_000
MARGIN           = 0.20
KIN_LK_SIGMA     = 3e-3
G_LW_SIGMA       = 1e-3
TEMP_SIGMA       = 15e-3
SEED             = 42

# realism knobs
COUPLING_DEPTH   = 0.85    # 1 → ideal short/open, 0 → no notch
INSERTION_LOSS   = 0.04    # linear scale (0.04 ≈ 0.36 dB)

# ── 1) RNG helper
cp.random.seed(SEED)
_normal = lambda mu, sig, shape: mu + sig * cp.random.standard_normal(shape, dtype=DTYPE)

# ── 2) Design data
try:
    f_nominal_GHz = cp.asarray(freqs, dtype=DTYPE)
except NameError:
    print("[Cell 6] Warning: 'freqs' not found – using demo sweep 5–8 GHz")
    f_nominal_GHz = cp.asarray([5.0, 6.0, 7.0, 8.0], dtype=DTYPE)

N_RES = int(f_nominal_GHz.size)

if "Q_L" in globals():
    Q_L_gpu = cp.asarray(Q_L, dtype=DTYPE)
    if Q_L_gpu.ndim == 0:
        Q_L_gpu = cp.repeat(Q_L_gpu, N_RES)
else:
    Q_L_gpu = cp.full(N_RES, 5_000, dtype=DTYPE)

# ── 3) Monte-Carlo parameter draws
_delta_Lk = _normal(0.0, KIN_LK_SIGMA, (NUM_PARTICLES, N_RES))
_delta_w  = _normal(0.0, G_LW_SIGMA,   (NUM_PARTICLES, N_RES))
_delta_T  = _normal(0.0, TEMP_SIGMA,   (NUM_PARTICLES, 1))

alpha = cp.asarray(-0.25,  dtype=DTYPE)   # geometry factor
beta  = cp.asarray(-1.8e-3, dtype=DTYPE)  # ppm / K for Nb 5–8 GHz

f_shifted = f_nominal_GHz[None, :] * (
      1.0 - 0.5*_delta_Lk + alpha*_delta_w + beta*_delta_T
)

# ── 4) Frequency grid
f_min   = cp.min(f_shifted) * (1 - MARGIN)
f_max   = cp.max(f_shifted) * (1 + MARGIN)
f_sweep = cp.linspace(f_min, f_max, N_SWEEP, dtype=DTYPE)      # (Ns,)

# ── 5) Notch Lorentzian helper
def lorentz_notch(f, f0, Q):
    """Magnitude of shunt-coupled λ/4 resonator with finite depth + loss."""
    x     = f / f0 - 1.0
    peak  = 1.0 / cp.sqrt(1.0 + (2.0 * Q * x)**2)
    return (1.0 - COUPLING_DEPTH * peak) * (1.0 - INSERTION_LOSS)

# ── 6) Allocate outputs
per_res_curves = cp.empty((N_RES, N_SWEEP), dtype=DTYPE)
global_env     = cp.full(N_SWEEP, cp.inf, dtype=DTYPE)

# ── 7) Streaming Monte-Carlo loop
for res_idx in range(N_RES):
    Qi      = Q_L_gpu[res_idx]
    fi_all  = f_shifted[:, res_idx]          # (Np,)
    res_min = cp.full(N_SWEEP, cp.inf, dtype=DTYPE)

    for start in range(0, NUM_PARTICLES, CHUNK_PARTICLES):
        end       = min(start + CHUNK_PARTICLES, NUM_PARTICLES)
        fi_block  = fi_all[start:end]                            # (block,)
        S_block   = lorentz_notch(f_sweep[:, None], fi_block[None, :], Qi)
        res_min   = cp.minimum(res_min, cp.min(S_block, axis=1))

    per_res_curves[res_idx] = res_min
    global_env = cp.minimum(global_env, res_min)

# ── 8) Plot in dB
fs_cpu   = cp.asnumpy(f_sweep)
to_dB    = lambda x: 20 * np.log10(x)
plt.figure(figsize=(10, 4))

for idx in range(N_RES):
    plt.plot(fs_cpu, to_dB(cp.asnumpy(per_res_curves[idx])),
             lw=0.8, label=f"Res {idx}")

plt.plot(fs_cpu, to_dB(cp.asnumpy(global_env)),
         lw=1.8, c="k", label="Global envelope")

plt.xlabel("Frequency (GHz)")
plt.ylabel("|S21|  (dB)")
plt.title("Predicted Per-Resonator |S21| Notches (GPU Monte-Carlo)")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, ncol=2, loc="upper right")
plt.tight_layout()
plt.show()

# ── 9) Optional CSV export
# import pandas as pd, os
# df = pd.DataFrame(to_dB(cp.asnumpy(per_res_curves.T)),
#                   columns=[f"Res_{i}" for i in range(N_RES)])
# df.insert(0, "GHz", fs_cpu)
# out_path = "per_resonator_S21_notch_curves_dB.csv"
# df.to_csv(out_path, index=False)
# print(f"Saved curves → {os.path.abspath(out_path)}")

In [ ]:
# Cell 7 — Monte-Carlo Frequency-Spread Density Map  (GPU → CPU)
# ─────────────────────────────────────────────────────────────────────────────
"""
Visualises the Monte-Carlo resonance-frequency spread produced in **Cell 6**.

Compatibility
─────────────
Relies on variables that already exist once Cell 6 has run:

    • f_shifted      – CuPy [NUM_PARTICLES, N_RES]  (MC-shifted frequencies)
    • NUM_PARTICLES  – integer
    • N_RES          – integer (len(freqs))          — falls back to f_shifted.shape[1]

If you renamed any of those, adjust the “Input hooks” section below.
"""

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np   # for Matplotlib only

# ── 0) Input hooks / fall-backs
if "f_shifted" not in globals():                      # sanity guard
    raise RuntimeError("f_shifted not found – run Cell 6 first.")

NUM_PARTICLES = globals().get("NUM_PARTICLES", f_shifted.shape[0])
N_RES         = globals().get("N_RES",          f_shifted.shape[1])

# ── 1) Parameters
BINS_FREQ  = 1024                     # horizontal resolution
BINS_INDEX = N_RES                    # one vertical bin per resonator
DTYPE      = cp.float32

# ── 2) Flatten data for histogram2d (GPU)
f_vals   = f_shifted.ravel().astype(DTYPE, copy=False)
idx_vals = cp.tile(cp.arange(N_RES, dtype=DTYPE), NUM_PARTICLES)

hist_gpu, f_edges, idx_edges = cp.histogram2d(
    f_vals, idx_vals, bins=[BINS_FREQ, BINS_INDEX]
)

# ── 3) Log-scale and ship small copy to CPU
hist_log = cp.log10(hist_gpu + 1)
hist_cpu      = cp.asnumpy(hist_log)          # (freq_bin, res_bin)
f_edges_cpu   = cp.asnumpy(f_edges)
idx_edges_cpu = cp.asnumpy(idx_edges)

# ── 4) Plot
plt.figure(figsize=(10, 3.2))
plt.imshow(
    hist_cpu.T, aspect="auto", origin="lower",
    extent=[float(f_edges_cpu[0]), float(f_edges_cpu[-1]),
            float(idx_edges_cpu[0]) - 0.5, float(idx_edges_cpu[-1]) - 0.5],
    cmap="viridis"
)
plt.colorbar(label="log₁₀ count")
plt.xlabel("Frequency (GHz)")
plt.ylabel("Resonator index")
plt.title("Monte-Carlo frequency-spread density map")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Resonator Statistics (ASCII-only, auto-detect + CuPy-aware)
# ─────────────────────────────────────────────────────────────────────────────
"""
Summarises per-resonator f₀, bandwidth, and loaded-Q in a pandas table,
then prints aggregate stats.

Works directly with the variables defined by Cells 1-6:

    • freqs      – 1-D NumPy **or** CuPy array [GHz]
    • Q_L        – scalar or 1-D array (optional)
    • bandwidths – optional scalar / array (same length as freqs)

Heuristics
──────────
* If the variable name doesn’t embed “GHz/MHz/kHz”, the script infers units:
      max(freqs) < 20  → GHz
      20 ≤ max(freqs) < 30 000 → MHz
      else → Hz
* If only Q and f₀ are available, bandwidth is calculated as Δf = f₀ / Q.
"""
# ---------- OPTIONAL MANUAL OVERRIDES
FREQ_VAR      = None   # e.g. "freqs"
BANDWIDTH_VAR = None   # e.g. "bandwidths_MHz"
Q_VAR         = None   # e.g. "Q_L"

# -------------------------------
import re, numbers, warnings
import numpy as np
import pandas as pd
try:
    import cupy as cp
except ModuleNotFoundError:
    cp = None
from IPython.display import display

UNIT_SCALE = {"ghz": 1e9, "mhz": 1e6, "khz": 1e3}

# ---------- helper functions

def unit_scale_from_name(name: str) -> float:
    """Return scale factor based on variable name suffix, else 1."""
    name = name.lower()
    for suf, scl in UNIT_SCALE.items():
        if suf in name:
            return scl
    return 1.0

def heuristic_freq_scale(arr: np.ndarray) -> float:
    """Infer scale if name gave us no clue."""
    mx = arr.max()
    if mx < 20.0:              # 3-10 GHz typical
        return 1e9
    if mx < 30_000.0:          # 20 MHz – 30 GHz
        return 1e6
    return 1.0                 # assume already Hz

def to_1d(arr_like):
    """Convert scalar / list / NumPy / CuPy to 1-D NumPy float array."""
    if isinstance(arr_like, numbers.Real):
        return np.asarray([arr_like], float)
    if cp and isinstance(arr_like, cp.ndarray):
        return cp.asnumpy(arr_like).ravel().astype(float)
    return np.asarray(arr_like, float).ravel()

def safe_len(obj):
    try:
        return len(obj)
    except Exception:
        return 0

# ---------- 1) locate frequency array
if FREQ_VAR:
    if FREQ_VAR not in globals():
        raise NameError(f"{FREQ_VAR!r} not defined.")
    freq_name = FREQ_VAR
    freq_arr  = to_1d(globals()[FREQ_VAR])
else:
    freq_name, freq_arr = None, None
    freq_pat = re.compile(r"(f0|freq|center.*f|res.*f)", re.I)
    for name, obj in list(globals().items()):
        if freq_pat.search(name) and safe_len(obj):
            freq_name = name
            freq_arr  = to_1d(obj)
            break
    if freq_arr is None:
        raise NameError("Frequency array not found – set FREQ_VAR.")

scale = unit_scale_from_name(freq_name)
if scale == 1.0:                       # ambiguous name → heuristic
    scale = heuristic_freq_scale(freq_arr)

f0_Hz = freq_arr * scale
N     = len(f0_Hz)

# ---------- 2) locate bandwidth or Q
bw_arr, q_arr = None, None
bw_name, q_name = None, None

# manual overrides first
if BANDWIDTH_VAR:
    if BANDWIDTH_VAR not in globals():
        raise NameError(f"{BANDWIDTH_VAR!r} not in workspace.")
    bw_name = BANDWIDTH_VAR
    bw_arr  = to_1d(globals()[BANDWIDTH_VAR])

if Q_VAR and bw_arr is None:
    if Q_VAR not in globals():
        raise NameError(f"{Q_VAR!r} not in workspace.")
    q_name = Q_VAR
    q_arr  = to_1d(globals()[Q_VAR])

# auto-detect if still missing
if bw_arr is None and q_arr is None:
    bw_pat = re.compile(r"(bw|bandwidth|deltaf|fwhm)", re.I)
    q_pat  = re.compile(r"(^q$|ql$|qvals|q_loaded|loadedq|q_l|qload)", re.I)

    for name, obj in list(globals().items()):
        if name == freq_name:
            continue
        L = safe_len(obj)
        if L not in (1, N):
            continue
        if bw_arr is None and bw_pat.search(name):
            bw_name, bw_arr = name, to_1d(obj)
        if q_arr is None and q_pat.search(name):
            q_name, q_arr = name, to_1d(obj)
        if bw_arr is not None or q_arr is not None:
            break

# broadcast scalars
if bw_arr is not None and len(bw_arr) == 1:
    bw_arr = np.repeat(bw_arr, N)
if q_arr is not None and len(q_arr) == 1:
    q_arr = np.repeat(q_arr, N)

if bw_arr is None and q_arr is None:
    candidates = [n for n, o in globals().items() if safe_len(o) in (1, N)]
    raise NameError(
        "No bandwidth or Q array matches frequency length.\n" +
        "Candidate arrays: " + ", ".join(candidates)
    )

# ---------- 3) compute missing quantities
if bw_arr is not None:
    scale_bw = unit_scale_from_name(bw_name)
    if scale_bw == 1.0:
        scale_bw = 1e6 if bw_arr.max() < 30_000 else 1.0
    bw_Hz    = bw_arr * scale_bw
    q_loaded = f0_Hz / bw_Hz
else:                                   # compute bandwidth from Q
    bw_Hz    = np.full_like(f0_Hz, np.nan)
    q_loaded = q_arr

# ---------- 4) build DataFrame & display
res_df = pd.DataFrame({
    "f0_GHz": f0_Hz / 1e9,
    "bandwidth_Hz": bw_Hz,
    "Q_loaded": q_loaded,
})

print(f"[info] Frequency   : '{freq_name}'  (scale={scale:g})")
if bw_arr is not None:
    print(f"[info] Bandwidth   : '{bw_name}'")
else:
    print(f"[info] Loaded-Q    : '{q_name}'")

print("\n▸ Per-resonator table:")
display(res_df)

print("▸ Aggregate statistics:")
agg = res_df[["f0_GHz", "bandwidth_Hz", "Q_loaded"]].agg(
    ["count", "mean", "std", "min", "max"]
).T
agg["mean ± σ"] = (
    agg["mean"].round(6).astype(str) + " ± " + agg["std"].round(6).astype(str)
)
display(agg[["count", "mean ± σ", "min", "max"]])